 ## 2: Kernel

In [2]:
# Jax configuration
USE_JIT = True
USE_X64 = True
DEBUG_NANS = False
VERBOSE = False

In [3]:
# Standard library imports
import os
os.environ['JAX_ENABLE_X64'] = str(USE_X64).lower()

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [4]:
# Third party
import jax
jax.config.update("jax_disable_jit", not USE_JIT)
jax.config.update("jax_debug_nans", DEBUG_NANS)

from jax import grad, vmap, jit
import jax.numpy as jnp
import jax.random as jr
import jax.lax as jlx
from optax import adam, apply_updates
from timeit import timeit

In [5]:
import numpy as np
from tqdm import tqdm
import logging
logging.basicConfig(level=logging.INFO)

In [6]:
from mimosa.linalg import cho_factor

In [7]:
from jax import numpy as jnp
from jax import Array
import equinox as eqx
from equinox import filter_jit

from kernax import AbstractKernel, StaticAbstractKernel, VarianceKernel, SEKernel

class StaticMOKernel(StaticAbstractKernel):
	@classmethod
	@filter_jit
	def pairwise_cov(cls, kern: AbstractKernel, x1: jnp.ndarray, x2: jnp.ndarray) -> jnp.ndarray:
		"""
		Compute the kernel covariance value between two vectors.

		:param kern: kernel instance containing the hyperparameters
		:param x1: scalar array
		:param x2: scalar array
		:return: scalar array
		"""
		kern = eqx.combine(kern)

		# As the formula only involves diagonal matrices, we can compute directly with vectors
		sigma_diag = jnp.exp(kern.length_scale_1) + jnp.exp(kern.length_scale_2) + jnp.exp(kern.length_scale_u)  # Σ
		sigma_det = jnp.prod(sigma_diag)  # |Σ|
		diff = x1 - x2  # x - x'

		# Compute the quadratic form: (x - x')^T Sigma^{-1} (x - x')
		# Since Sigma^{-1} is diagonal, this simplifies to sum of (diff_i^2 * sigma_inv_diag_i)
		quadratic_form = jnp.sum(diff**2 / sigma_diag)

		return jnp.exp(kern.variance_1) * jnp.exp(kern.variance_2) /(((2 * jnp.pi)**(len(x1)/2)) * jnp.sqrt(sigma_det)) * jnp.exp(-0.5 * quadratic_form)


class MOKernel(AbstractKernel):
	"""
	Squared Exponential (aka "RBF" or "Gaussian") Kernel
	"""

	length_scale_1: Array = eqx.field(converter=jnp.asarray)
	length_scale_2: Array = eqx.field(converter=jnp.asarray)
	length_scale_u: Array = eqx.field(converter=jnp.asarray)
	variance_1: Array = eqx.field(converter=jnp.asarray)
	variance_2: Array = eqx.field(converter=jnp.asarray)

	static_class = StaticMOKernel

	def __init__(self, length_scale_1, length_scale_2, length_scale_u, variance_1, variance_2):
		super().__init__()
		self.length_scale_1 = length_scale_1
		self.length_scale_2 = length_scale_2
		self.length_scale_u = length_scale_u
		self.variance_1 = variance_1
		self.variance_2 = variance_2

---

## Dataset

In [8]:
nb_samples = 100
nb_blocks = 2

In [9]:
vrs = jnp.linspace(.1, 1., 20)
lss = jnp.linspace(-5., 2.5, 20)
lus = jnp.linspace(-7., 3.5, 20)
inputs = jnp.linspace(-3, 3, 100)[:, None]

INFO:2026-03-11 23:15:10,178:jax._src.xla_bridge:834: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
2026-03-11 23:15:10,178 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [10]:
key = jr.PRNGKey(42)
warm_up_key, test_key = jr.split(key)
warm_up_keys = jr.split(warm_up_key, nb_samples)
test_keys = jr.split(test_key, nb_samples)
warm_up_keys.shape, test_keys.shape

((100, 2), (100, 2))

In [11]:
@filter_jit
def single_block_cov(key, nb_blocks):
	keys = jr.split(key, (3,))
	actual_vrs = jr.choice(keys[0], vrs, (nb_blocks,))
	actual_lss = jr.choice(keys[1], lss, (nb_blocks,))
	actual_lu = jr.choice(keys[2], lus)

	rows = []
	for row_kern in range(nb_blocks):
		row = []
		for col_kern in range(nb_blocks):
			sub_block = MOKernel(
				length_scale_1=actual_lss[row_kern],
				length_scale_2=actual_lss[col_kern],
				length_scale_u=actual_lu,
				variance_1=actual_vrs[row_kern],
				variance_2=actual_vrs[col_kern])(inputs)
			row.append(sub_block)
		rows.append(row)

	return jnp.block(rows)

In [12]:
single_block_cov(jr.PRNGKey(42), 4).shape

(400, 400)

In [13]:
warm_up_covs = vmap(lambda k: single_block_cov(k, nb_blocks))(warm_up_keys)
test_covs = vmap(lambda k: single_block_cov(k, nb_blocks))(test_keys)
warm_up_covs.shape, test_covs.shape

((100, 200, 200), (100, 200, 200))

In [14]:
np.asarray(warm_up_covs[0])

array([[1.00069356e+00, 9.98992583e-01, 9.93906972e-01, ...,
        4.86121185e-11, 2.95630403e-11, 1.78870328e-11],
       [9.98992583e-01, 1.00069356e+00, 9.98992583e-01, ...,
        7.95288429e-11, 4.86121185e-11, 2.95630403e-11],
       [9.93906972e-01, 9.98992583e-01, 1.00069356e+00, ...,
        1.29446240e-10, 7.95288429e-11, 4.86121185e-11],
       ...,
       [4.86121185e-11, 7.95288429e-11, 1.29446240e-10, ...,
        1.90344290e+00, 1.89377366e+00, 1.86505968e+00],
       [2.95630403e-11, 4.86121185e-11, 7.95288429e-11, ...,
        1.89377366e+00, 1.90344290e+00, 1.89377366e+00],
       [1.78870328e-11, 2.95630403e-11, 4.86121185e-11, ...,
        1.86505968e+00, 1.89377366e+00, 1.90344290e+00]], shape=(200, 200))

In [15]:
jnp.isnan(warm_up_covs).any(), jnp.isnan(test_covs).any()

(Array(False, dtype=bool), Array(False, dtype=bool))

---

## Classical approach:

In [16]:
from mimosa.linalg import cho_factor, cho_solve

In [57]:
# Warm-up
fac = cho_factor(warm_up_covs)
fac.shape, jnp.isnan(fac).any()

((100, 200, 200), Array(False, dtype=bool))

In [58]:
%timeit cho_factor(test_covs).block_until_ready()

3.59 ms ± 36.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


---

## Custom function - iterative

In [19]:
# Check that covs are block-symmetric
for i in range(nb_blocks):
	for j in range(nb_blocks):
		print(jnp.isclose(warm_up_covs[0][i*100:(i+1)*100, j*100:(j+1)*100], warm_up_covs[0][j*100:(j+1)*100, i*100:(i+1)*100].T).all())

True
True
True
True


In [20]:
jnp.triu_indices(4)

(Array([0, 0, 0, 0, 1, 1, 1, 2, 2, 3], dtype=int64),
 Array([0, 1, 2, 3, 1, 2, 3, 2, 3, 3], dtype=int64))

In [21]:
def full_to_batched(cov: Array, nb_blocks, nb_points: int) -> Array:
	return vmap(lambda i, j: jax.lax.dynamic_slice(cov, (i*nb_points, j*nb_points), (nb_points, nb_points)))(*jnp.triu_indices(nb_blocks))

In [43]:
warm_up_covs_batched = vmap(lambda c: full_to_batched(c, nb_blocks, 100))(warm_up_covs)
test_covs_batched = vmap(lambda c: full_to_batched(c, nb_blocks, 100))(test_covs)
warm_up_covs_batched.shape, test_covs_batched.shape

((100, 3, 100, 100), (100, 3, 100, 100))

In [22]:
full_to_batched(test_covs[0], nb_blocks, 100).shape

(3, 100, 100)

In [23]:
np.asarray(test_covs[0])

array([[1.51651967e+00, 1.51316719e+00, 1.50315413e+00, ...,
        1.23267543e-15, 5.98509212e-16, 2.88452764e-16],
       [1.51316719e+00, 1.51651967e+00, 1.51316719e+00, ...,
        2.52004562e-15, 1.23267543e-15, 5.98509212e-16],
       [1.50315413e+00, 1.51316719e+00, 1.51651967e+00, ...,
        5.11387175e-15, 2.52004562e-15, 1.23267543e-15],
       ...,
       [1.23267543e-15, 2.52004562e-15, 5.11387175e-15, ...,
        2.58723397e+00, 2.55797862e+00, 2.47218250e+00],
       [5.98509212e-16, 1.23267543e-15, 2.52004562e-15, ...,
        2.55797862e+00, 2.58723397e+00, 2.55797862e+00],
       [2.88452764e-16, 5.98509212e-16, 1.23267543e-15, ...,
        2.47218250e+00, 2.55797862e+00, 2.58723397e+00]], shape=(200, 200))

In [24]:
np.asarray(full_to_batched(test_covs[0], nb_blocks, 100)[2])

array([[2.58723397e+00, 2.55797862e+00, 2.47218250e+00, ...,
        8.78462877e-47, 9.56441899e-48, 1.01792595e-48],
       [2.55797862e+00, 2.58723397e+00, 2.55797862e+00, ...,
        7.88697831e-46, 8.78462877e-47, 9.56441899e-48],
       [2.47218250e+00, 2.55797862e+00, 2.58723397e+00, ...,
        6.92181983e-45, 7.88697831e-46, 8.78462877e-47],
       ...,
       [8.78462877e-47, 7.88697831e-46, 6.92181983e-45, ...,
        2.58723397e+00, 2.55797862e+00, 2.47218250e+00],
       [9.56441899e-48, 8.78462877e-47, 7.88697831e-46, ...,
        2.55797862e+00, 2.58723397e+00, 2.55797862e+00],
       [1.01792595e-48, 9.56441899e-48, 8.78462877e-47, ...,
        2.47218250e+00, 2.55797862e+00, 2.58723397e+00]], shape=(100, 100))

In [25]:
jnp.array([])

Array([], shape=(0,), dtype=float64)

In [26]:
test = jnp.ones((4, 3, 2, 5))
test.shape, test.T.shape, test.mT.shape

((4, 3, 2, 5), (5, 2, 3, 4), (4, 3, 5, 2))

In [212]:
def test():
	return 1, 2, 3, 4

*_, c, d = test()

c, d

(3, 4)

In [38]:
@jit
def block2_cho_factor(arr: Array) -> Array:
	N = arr.shape[0]//2
	factor = jnp.zeros_like(arr)

	# Set F_11 to chol(A_11)
	factor = factor.at[:N, :N].set(cho_factor(arr[:N, :N]))

	# Solve F_21 @ F_11^T = A_21 to find F_21
	factor = factor.at[:N, N:].set(jlx.linalg.triangular_solve(factor[:N, :N], arr[:N, N:]).T)

	# Set F_22 to chol(A_22 - F_21 @ F_21^T)
	factor = factor.at[N:, N:].set(cho_factor(arr[N:, N:] - (factor[:N, N:].T @ factor[:N, N:])))

	return factor

In [59]:
@jit
def block2_cho_factor(arr: Array) -> Array:
	# Set F_11 to chol(A_11)
	arr = arr.at[0].set(cho_factor(arr[0]))

	# Solve F_21 @ F_11^T = A_21 to find F_21
	arr = arr.at[1].set(jlx.linalg.triangular_solve(arr[0], arr[1]).T)

	# Set F_22 to chol(A_22 - F_21 @ F_21^T)
	arr = arr.at[2].set(cho_factor(arr[2] - (arr[1].T @ arr[1])))

	return arr

In [46]:
block2_cho_factor(test_covs_batched[0])

Array([[[ 1.23147459e+00,  1.22874414e+00,  1.22061319e+00, ...,
          1.11460924e-09,  7.23938568e-10,  4.68121430e-10],
        [ 0.00000000e+00,  8.19604344e-02,  1.62833183e-01, ...,
          8.96085257e-09,  5.89403526e-09,  3.85930714e-09],
        [ 0.00000000e+00,  0.00000000e+00,  1.08836762e-02, ...,
          3.59476205e-08,  2.39451676e-08,  1.58764831e-08],
        ...,
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          5.85427913e-03,  7.11015738e-03,  1.11313340e-02],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  5.85422968e-03,  7.10998068e-03],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  5.85416089e-03]],

       [[ 1.38232192e+00,  1.37720972e+00,  1.36198626e+00, ...,
          1.00097512e-15,  4.86010200e-16,  2.34233630e-16],
        [-3.07610085e-02,  1.22692839e-01,  2.74109604e-01, ...,
          1.57405621e-14,  7.75365879e

In [51]:
jnp.allclose(jnp.triu(block2_cho_factor(test_covs_batched[0])), cho_factor(test_covs[0]))

ValueError: Incompatible shapes for broadcasting: shapes=[(3, 100, 100), (200, 200)]

In [52]:
jnp.triu(block2_cho_factor(test_covs_batched[0])).shape

(3, 100, 100)

In [60]:
batched_block2_cho_factor = jit(vmap(block2_cho_factor))

In [61]:
# Warm-up
fac = batched_block2_cho_factor(warm_up_covs_batched)
fac.shape, jnp.isnan(fac).any()

((100, 3, 100, 100), Array(False, dtype=bool))

In [62]:
%timeit batched_block2_cho_factor(test_covs_batched).block_until_ready()

6.45 ms ± 37.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# Warm up


In [ ]:
@filter_jit
def block_cho_factor(arr: Array, nb_blocks: int) -> Array:
	chol = jnp.zeros_like(arr)

	for j in range(nb_blocks):
		for i in range(j, nb_blocks):
			# Compute S = A_ij - sum_k(G_ik * G_jk^T)
			s = arr[]

## PSD sub-blocks do not guarantee PSD block matrices

In [7]:
inputs = jnp.array([[1.], [2.]])

INFO:2026-03-09 08:48:24,349:jax._src.xla_bridge:834: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
2026-03-09 08:48:24,349 - INFO - Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/miniconda3/envs/mimosa/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [8]:
k1 = VarianceKernel(variance=.5) * SEKernel(length_scale=.1)
k2 = VarianceKernel(variance=10.) * SEKernel(length_scale=1.)

### Build each block

In [9]:
A = k1(inputs)
B = k2(inputs)
C = k1(inputs)
full_cov = jnp.block([[A, B], [B.T, C]])
np.asarray(full_cov)

array([[5.00000000e-01, 9.64374924e-23, 1.00000000e+01, 6.06530660e+00],
       [9.64374924e-23, 5.00000000e-01, 6.06530660e+00, 1.00000000e+01],
       [1.00000000e+01, 6.06530660e+00, 5.00000000e-01, 9.64374924e-23],
       [6.06530660e+00, 1.00000000e+01, 9.64374924e-23, 5.00000000e-01]])

### Check if each block is PSD

In [10]:
np.asarray(cho_factor(A))

array([[7.07113852e-01, 1.36381846e-22],
       [0.00000000e+00, 7.07113852e-01]])

In [11]:
np.asarray(cho_factor(B))

array([[3.16227924, 1.9180174 ],
       [0.        , 2.51420351]])

In [12]:
np.asarray(cho_factor(C))

array([[7.07113852e-01, 1.36381846e-22],
       [0.00000000e+00, 7.07113852e-01]])

### But the full block matrix is not PSD, no matter the jitter

In [13]:
jitter = 1e-3

In [14]:
cho_factor(full_cov + jitter * jnp.eye(full_cov.shape[0]))

Array([[nan, nan, nan, nan],
       [ 0., nan, nan, nan],
       [ 0.,  0., nan, nan],
       [ 0.,  0.,  0., nan]], dtype=float64)

In [15]:
jnp.isnan(cho_factor(full_cov + jitter * jnp.eye(full_cov.shape[0]))).any().item()

True

Is the min eigenvalue of C smaller than the product of the min eigenvalues of A and B**2? If so, the full block matrix cannot be PSD.

In [16]:
print(f"{jnp.real(jnp.linalg.eigvals(C)).min().item()} >?= {jnp.real(jnp.linalg.eigvals(A)).min().item()} * {(jnp.real(jnp.linalg.eigvals(B)).min()**2).item()}")

0.5 >?= 0.5 * 15.48181217461755


## Can the MO kernel produce non-PSD block matrices?

In [17]:
vrs = jnp.linspace(.01, 15., 10)
lss = jnp.linspace(-5., 5., 15)
lus = jnp.linspace(-10., 10., 10)
inputs = jnp.linspace(-3, 3, 100)[:, None]

In [21]:
for lu in lus:
	for var1 in vrs:
		for ls1 in lss:
			for ls2 in lss:
				for var2 in vrs:
					A = MOKernel(length_scale_1=ls1, length_scale_2=ls1, length_scale_u=lu, variance_1=var1, variance_2=var1)(inputs)
					B = MOKernel(length_scale_1=ls1, length_scale_2=ls2, length_scale_u=lu, variance_1=var1, variance_2=var2)(inputs)
					C = MOKernel(length_scale_1=ls2, length_scale_2=ls2, length_scale_u=lu, variance_1=var2, variance_2=var2)(inputs)

					psd_A = not jnp.isnan(cho_factor(A + jitter*jnp.eye(A.shape[0]))).any().item()
					psd_B = not jnp.isnan(cho_factor(B + jitter*jnp.eye(B.shape[0]))).any().item()
					psd_C = not jnp.isnan(cho_factor(C + jitter*jnp.eye(C.shape[0]))).any().item()

					K = jnp.block([[A, B], [B.T, C]])

					psd_K = not jnp.isnan(cho_factor(K + jitter * jnp.eye(K.shape[0]))).any().item()

					if not psd_K:
						if psd_A and psd_B and psd_C:
							logging.error(f"Non-PSD block matrix with PSD blocks for ls1={ls1:.2f}, ls2={ls2:.2f}, lu={lu:.2f}, var1={var1:.2f}, var2={var2:.2f}")
						else:
							logging.warning(f"Non-PSD block matrix with non-PSD blocks for ls1={ls1:.2f}, ls2={ls2:.2f}, lu={lu:.2f}, var1={var1:.2f}, var2={var2:.2f}")
					else:
						logging.info(f"PSD block matrix for ls1={ls1:.2f}, ls2={ls2:.2f}, lu={lu:.2f}, var1={var1:.2f}, var2={var2:.2f}")

2026-03-08 22:22:30,964 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=0.01
2026-03-08 22:22:30,967 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=1.67
2026-03-08 22:22:30,968 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=3.34
2026-03-08 22:22:30,970 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=5.00
2026-03-08 22:22:30,972 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=6.67
2026-03-08 22:22:30,973 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=8.33
2026-03-08 22:22:30,975 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.30, lu=-2.00, var1=0.01, var2=10.00
2026-03-08 22:22:30,977 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.01, lu=-2.00, var1=0.01, var2=0.01
2026-03-08 22:22:30,979 - INFO - PSD block matrix for ls1=-1.30, ls2=-1.01, lu=-2.00, var1=0.01, var2=1.67
2026-03-08 22:22:30,981 - INFO - PSD

### Example from the MO kernel

In [76]:
#ls1=4.29; ls2=2.86; lu=-10.00; var1=0.01; var2=15.00
ls1=-2.47; ls2=4.25; lu=-3.72; var1=-4.23; var2=13.57

In [77]:
A = MOKernel(length_scale_1=ls1, length_scale_2=ls1, length_scale_u=lu, variance_1=var1, variance_2=var1)(inputs)
B = MOKernel(length_scale_1=ls1, length_scale_2=ls2, length_scale_u=lu, variance_1=var1, variance_2=var2)(inputs)
C = MOKernel(length_scale_1=ls2, length_scale_2=ls2, length_scale_u=lu, variance_1=var2, variance_2=var2)(inputs)

In [78]:
np.asarray(A)

array([[1.92108409e-04, 1.90292796e-04, 1.84948265e-04, ...,
        3.02384958e-43, 4.74651201e-44, 7.31039641e-45],
       [1.90292796e-04, 1.92108409e-04, 1.90292796e-04, ...,
        1.89015634e-42, 3.02384958e-43, 4.74651201e-44],
       [1.84948265e-04, 1.90292796e-04, 1.92108409e-04, ...,
        1.15927698e-41, 1.89015634e-42, 3.02384958e-43],
       ...,
       [3.02384958e-43, 1.89015634e-42, 1.15927698e-41, ...,
        1.92108409e-04, 1.90292796e-04, 1.84948265e-04],
       [4.74651201e-44, 3.02384958e-43, 1.89015634e-42, ...,
        1.90292796e-04, 1.92108409e-04, 1.90292796e-04],
       [7.31039641e-45, 4.74651201e-44, 3.02384958e-43, ...,
        1.84948265e-04, 1.90292796e-04, 1.92108409e-04]], shape=(100, 100))

In [79]:
np.asarray(B)

array([[542.0108176 , 541.99664077, 541.95411249, ..., 423.76580609,
        421.60989718, 419.4430137 ],
       [541.99664077, 542.0108176 , 541.99664077, ..., 425.91045817,
        423.76580609, 421.60989718],
       [541.95411249, 541.99664077, 542.0108176 , ..., 428.04357151,
        425.91045817, 423.76580609],
       ...,
       [423.76580609, 425.91045817, 428.04357151, ..., 542.0108176 ,
        541.99664077, 541.95411249],
       [421.60989718, 423.76580609, 425.91045817, ..., 541.99664077,
        542.0108176 , 541.99664077],
       [419.4430137 , 421.60989718, 423.76580609, ..., 541.95411249,
        541.99664077, 542.0108176 ]], shape=(100, 100))

In [80]:
np.asarray(C)

array([[2.06174051e+10, 2.06171351e+10, 2.06163251e+10, ...,
        1.82271715e+10, 1.81806831e+10, 1.81338382e+10],
       [2.06171351e+10, 2.06174051e+10, 2.06171351e+10, ...,
        1.82733002e+10, 1.82271715e+10, 1.81806831e+10],
       [2.06163251e+10, 2.06171351e+10, 2.06174051e+10, ...,
        1.83190658e+10, 1.82733002e+10, 1.82271715e+10],
       ...,
       [1.82271715e+10, 1.82733002e+10, 1.83190658e+10, ...,
        2.06174051e+10, 2.06171351e+10, 2.06163251e+10],
       [1.81806831e+10, 1.82271715e+10, 1.82733002e+10, ...,
        2.06171351e+10, 2.06174051e+10, 2.06171351e+10],
       [1.81338382e+10, 1.81806831e+10, 1.82271715e+10, ...,
        2.06163251e+10, 2.06171351e+10, 2.06174051e+10]], shape=(100, 100))

In [81]:
K = jnp.block([[A, B], [B.T, C]])
np.asarray(K)

array([[1.92108409e-04, 1.90292796e-04, 1.84948265e-04, ...,
        4.23765806e+02, 4.21609897e+02, 4.19443014e+02],
       [1.90292796e-04, 1.92108409e-04, 1.90292796e-04, ...,
        4.25910458e+02, 4.23765806e+02, 4.21609897e+02],
       [1.84948265e-04, 1.90292796e-04, 1.92108409e-04, ...,
        4.28043572e+02, 4.25910458e+02, 4.23765806e+02],
       ...,
       [4.23765806e+02, 4.25910458e+02, 4.28043572e+02, ...,
        2.06174051e+10, 2.06171351e+10, 2.06163251e+10],
       [4.21609897e+02, 4.23765806e+02, 4.25910458e+02, ...,
        2.06171351e+10, 2.06174051e+10, 2.06171351e+10],
       [4.19443014e+02, 4.21609897e+02, 4.23765806e+02, ...,
        2.06163251e+10, 2.06171351e+10, 2.06174051e+10]], shape=(200, 200))

In [82]:
print(f"Minimum eigvalue from A = {jnp.real(jnp.linalg.eigvals(A + jitter*jnp.eye(A.shape[0]))).min().item()}")
print(f"Minimum eigvalue from B = {jnp.real(jnp.linalg.eigvals(B + jitter*jnp.eye(B.shape[0]))).min().item()}")
print(f"Minimum eigvalue from C = {jnp.real(jnp.linalg.eigvals(C + jitter*jnp.eye(C.shape[0]))).min().item()}")
print(f"Minimum eigvalue from K = {jnp.real(jnp.linalg.eigvals(K + jitter*jnp.eye(K.shape[0]))).min().item()}")

Minimum eigvalue from A = 0.0009999999999999976
Minimum eigvalue from B = 0.000999999999301908
Minimum eigvalue from C = 0.0009661309884218658
Minimum eigvalue from K = 0.0009762720992465159


In [83]:
print(f"Minimum eigvalue from C = {jnp.real(jnp.linalg.eigvals(C + jitter*jnp.eye(C.shape[0]))).min().item()} >?= {jnp.real(jnp.linalg.eigvals(A + jitter*jnp.eye(A.shape[0]))).min().item()} * {(jnp.real(jnp.linalg.eigvals(B + jitter*jnp.eye(B.shape[0]))).min()**2).item()}")
print(f"Minimum eigvalue from C = {jnp.real(jnp.linalg.eigvals(C + jitter*jnp.eye(C.shape[0]))).min().item()} >?= {jnp.real(jnp.linalg.eigvals(A + jitter*jnp.eye(A.shape[0]))).min().item() * (jnp.real(jnp.linalg.eigvals(B + jitter*jnp.eye(B.shape[0]))).min()**2).item()}")

Minimum eigvalue from C = 0.0009661309884218658 >?= 0.0009999999999999976 * 9.99999998603816e-07
Minimum eigvalue from C = 0.0009661309884218658 >?= 9.999999986038137e-10


In [84]:
np.asarray(cho_factor(K + 1e-1 * jnp.eye(K.shape[0])))

array([[3.16547166e-01, 6.01151475e-04, 5.84267637e-04, ...,
        1.33871300e+03, 1.33190230e+03, 1.32505692e+03],
       [0.00000000e+00, 3.16546595e-01, 6.00042980e-04, ...,
        1.34294823e+03, 1.33618601e+03, 1.32938829e+03],
       [0.00000000e+00, 0.00000000e+00, 3.16546058e-01, ...,
        1.34721494e+03, 1.34050161e+03, 1.33375197e+03],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        3.88963218e-01, 1.60867684e-01, 1.93740710e-01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 3.88901995e-01, 1.60803201e-01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 3.88936919e-01]], shape=(200, 200))

## Optimise MO kernel to have a non-PSD block matrix

In [63]:
def criteria(hps, inputs):
	# hps = [ls1, ls2, lu, var1, var2]
	ls1, ls2, lu, var1, var2 = hps
	A = MOKernel(length_scale_1=ls1, length_scale_2=ls1, length_scale_u=lu, variance_1=var1, variance_2=var1)(inputs)
	B = MOKernel(length_scale_1=ls1, length_scale_2=ls2, length_scale_u=lu, variance_1=var1, variance_2=var2)(inputs)
	C = MOKernel(length_scale_1=ls2, length_scale_2=ls2, length_scale_u=lu, variance_1=var2, variance_2=var2)(inputs)

	eig_A_min = jnp.real(jnp.linalg.eigvals(A)).min()
	eig_B_min = jnp.real(jnp.linalg.eigvals(B)).min()
	eig_C_min = jnp.real(jnp.linalg.eigvals(C)).min()

	# We want A, B and C to be PSD, so we maximise their minimum eigenvalues
	indiv_PSDs =  - (jnp.exp(eig_A_min) + jnp.exp(eig_B_min) + jnp.exp(eig_C_min))

	# We want K to be non-PSD => eig_C_min < eig_A_min * eig_B_min**2 => we minimise the difference between the right and left hand sides of this inequality
	block_non_PSD = - (jnp.exp(eig_A_min * eig_B_min**2 - eig_C_min))

	return indiv_PSDs + block_non_PSD

In [71]:
ls1=-2.86; ls2=3.57; lu=-9.00; var1=1.68; var2=13.33
hps = jnp.array([ls1, ls2, lu, var1, var2])
hps

Array([-2.86,  3.57, -9.  ,  1.68, 13.33], dtype=float64)

In [72]:
print('Objective function: ', criteria(hps, inputs))

Objective function:  -4.000000000287894


In [73]:
print('Gradients: ', grad(criteria)(hps, inputs))

Gradients:  [-1.13449063e-13  4.20619983e-10 -2.82221247e-17  1.25550567e-10
 -1.33448452e-09]


In [74]:
solver = adam(learning_rate=1.)
opt_state = solver.init(hps)

In [75]:
for e in range(10000):
	grads = grad(criteria)(hps, inputs)
	updates, opt_state = solver.update(grads, opt_state, hps)
	hps = apply_updates(hps, updates)
	if e % 50 == 0:
		print(f'Epoch {e:3} | Objective function: {criteria(hps, inputs):.2E}')
		print(f"\tls1={hps[0]:.2f}, ls2={hps[1]:.2f}, lu={hps[2]:.2f}, var1={hps[3]:.2f}, var2={hps[4]:.2f}")
		#print(f"\tA: {"PSD" if not jnp.isnan(cho_factor(a(inputs) + jitter*jnp.eye(inputs.shape[0]))).any().item() else "non-PSD"}, B: {"PSD" if not jnp.isnan(cho_factor(b(inputs) + jitter*jnp.eye(inputs.shape[0]))).any().item() else "non-PSD"}, C: {"PSD" if not jnp.isnan(cho_factor(c(inputs) + jitter*jnp.eye(inputs.shape[0]))).any().item() else "non-PSD"}")
		#print(f"\tK: {"PSD" if not jnp.isnan(cho_factor(jnp.block([[a(inputs), b(inputs)], [b(inputs).T, c(inputs)]]) + jitter*jnp.eye(2*inputs.shape[0]))).any().item() else "non-PSD"}")

Epoch   0 | Objective function: -4.00E+00
	ls1=-2.86, ls2=3.53, lu=-9.00, var1=1.67, var2=13.45
Epoch  50 | Objective function: -4.00E+00
	ls1=-2.47, ls2=4.25, lu=-3.72, var1=-4.23, var2=13.57
Epoch 100 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.38, var2=13.28
Epoch 150 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.38, var2=13.28
Epoch 200 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.38, var2=13.28
Epoch 250 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.38, var2=13.28
Epoch 300 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.38, var2=13.28
Epoch 350 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.38, var2=13.28
Epoch 400 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.39, var2=13.28
Epoch 450 | Objective function: -4.00E+00
	ls1=-2.45, ls2=4.55, lu=-3.42, var1=-4.39, var2=13.28
Epoch 500 | Objective function:

KeyboardInterrupt: 